In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv .env.furflex.demo

In [ ]:
from mlde_analysis.default_params import *
variable = "pr"
domain = "birmingham-64"
frequency = "1hr"
scenario = "rcp85"
collection = "land-cpm"
resolution = "2.2km-coarsened-4x"

In [ ]:
import functools
import math
import string

import IPython
from IPython.display import HTML
import matplotlib
from matplotlib import animation
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import os
import cf_xarray

from mlde_utils import cp_model_rotated_pole, VariableMetadata
from mlde_analysis import DERIVED_DATA, plot_map
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.data import si_to_mmhour

In [ ]:
ds = xr.open_dataset(
    VariableMetadata(
        base_dir=DERIVED_DATA/"moose",
        variable=variable,
        domain=domain,
        frequency=frequency,
        resolution=resolution,
        scenario=scenario,
        collection=collection,
        ensemble_member="01",
    ).filepath(1981)
).isel(time=slice(0,24))
ds["pr"] = si_to_mmhour(ds["pr"])
ds

In [ ]:
fig, ax = plt.subplots(figsize=(4,4), subplot_kw={"projection": cp_model_rotated_pole})

da_sel = ds[variable]
# data0 = da_sel.isel(time=0).values
# im = ax.imshow(data0, origin='lower')#, cmap=cmap, extent=extent)
# cbar = fig.colorbar(im, ax=ax)

quad  = plot_map(da_sel.isel(time=0), ax=ax, style="pr")
title = ax.set_title(str(ds['time'].values[0]))

def update(frame):
    # ax.collections.clear()
    quad = plot_map(da_sel.isel(time=frame), ax=ax, style="pr")
    title.set_text(str(da_sel['time'].values[frame]))
    return (quad, title)

anim = animation.FuncAnimation(fig, update, frames=24, blit=False)
plt.close(fig)  # prevent double display in notebook
# Display as HTML5 video (works in JupyterLab)
# HTML(anim.to_jshtml())
HTML(anim.to_html5_video())